# NumPy Fundamentals for Analysts

**Instructor**: Patrick Dolinger

**Generated**: 2025-12-04

This notebook is a hands-on tutorial that introduces NumPy (and a quick pandas tie-in) for data analysts.

**What you'll learn:**
- pandas and NumPy intro
- NumPy arrays and array properties
- Array basics & creation
- Random number generation (modern `default_rng`)
- Indexing & slicing
- Array operations & ufuncs
- Filtering, modifying values, and `np.where`
- Aggregation & sorting
- Vectorization & broadcasting

**Tip:** Run cells in order. If a section mentions *view vs copy*, check the examples carefully.


## Setup & Versions

In [1]:
import numpy as np
import pandas as pd

# Prefer the modern Generator API for randomness
rng = np.random.default_rng(seed=42)

print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")


NumPy: 2.3.5
pandas: 2.3.3


## pandas and NumPy intro

NumPy provides fast, contiguous arrays; pandas builds on NumPy to offer labeled, tabular structures. Convert between them using `.to_numpy()` or the `.values` attribute.


In [2]:
# Create a small pandas DataFrame
df = pd.DataFrame({
    'region': ['NS', 'NS', 'NB', 'PE'],
    'sales': [100, 140, 90, 120],
    'margin': [0.30, 0.28, 0.25, 0.33]
})

# Convert numeric columns to NumPy arrays
sales_np = df['sales'].to_numpy()
margin_np = df['margin'].to_numpy(dtype=np.float64)

print("sales_np:", sales_np, type(sales_np), sales_np.dtype)
print("margin_np:", margin_np, type(margin_np), margin_np.dtype)

# Back to pandas using Index labels
arr = np.array([[1,2,3],[4,5,6]], dtype=np.int64)
back_df = pd.DataFrame(arr, columns=['A','B','C'])
back_df


sales_np: [100 140  90 120] <class 'numpy.ndarray'> int64
margin_np: [0.3  0.28 0.25 0.33] <class 'numpy.ndarray'> float64


,A,B,C
0,1,2,3
1,4,5,6


## NumPy arrays and array properties

In [3]:
a = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int32)
print("ndim:", a.ndim)       # number of dimensions
print("shape:", a.shape)    # rows, cols
print("size:", a.size)      # total elements
print("dtype:", a.dtype)
print("itemsize:", a.itemsize, "bytes per element")
print("nbytes:", a.nbytes, "total bytes")

# Change dtype (careful with precision)
a_f = a.astype(np.float64)
print("converted dtype:", a_f.dtype)


ndim: 2
shape: (2, 3)
size: 6
dtype: int32
itemsize: 4 bytes per element
nbytes: 24 total bytes
converted dtype: float64


## Array basics

In [4]:
# Creating from Python lists
vec = np.array([10, 20, 30, 40])
print(vec)

# 2D array
mat = np.array([[1, 2], [3, 4], [5, 6]])
print(mat)

# Basic indexing
print("vec[0] ->", vec[0])
print("mat[1, 0] ->", mat[1, 0])

# Slicing returns a *view* for basic slices
slice_view = vec[1:3]
print("slice_view:", slice_view)
slice_view[0] = -999
print("After modifying slice_view, original vec:", vec)

# Copy explicitly if you need independence
slice_copy = vec[1:3].copy()
slice_copy[0] = 777
print("slice_copy:", slice_copy)
print("vec unchanged:", vec)


[10 20 30 40]
[[1 2]
 [3 4]
 [5 6]]
vec[0] -> 10
mat[1, 0] -> 3
slice_view: [20 30]
After modifying slice_view, original vec: [  10 -999   30   40]
slice_copy: [777  30]
vec unchanged: [  10 -999   30   40]


## Array creation

In [5]:
print("arange:", np.arange(0, 10, 2))
print("linspace:", np.linspace(0, 1, 5))

Z = np.zeros((2,3), dtype=np.float32)
O = np.ones((2,3), dtype=np.int16)
F = np.full((2,3), fill_value=7)
E = np.empty((2,3))  # values are undefined; for performance
I = np.eye(3)        # identity matrix
D = np.diag([1,2,3]) # diagonal from 1D

Z, O, F, E, I, D


arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]


(array([[0., 0., 0.],
        [0., 0., 0.]], dtype=float32),
 array([[1, 1, 1],
        [1, 1, 1]], dtype=int16),
 array([[7, 7, 7],
        [7, 7, 7]]),
 array([[1.15540371e-311, 1.15540371e-311, 1.15540371e-311],
        [1.15540371e-311, 1.15540371e-311, 1.15540371e-311]]),
 array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]),
 array([[1, 0, 0],
        [0, 2, 0],
        [0, 0, 3]]))

## Random number generation (modern API)

In [6]:
# Using Generator for reproducibility
rng = np.random.default_rng(12345)
print(rng.normal(loc=0, scale=1, size=5))
print(rng.integers(low=0, high=10, size=(2,3)))
print(rng.uniform(0, 1, size=4))

# Shuffle and choice
arr = np.arange(10)
rng.shuffle(arr)
print("shuffled:", arr)
print("choice:", rng.choice(arr, size=3, replace=False))


[-1.42382504  1.26372846 -0.87066174 -0.25917323 -0.07534331]
[[8 3 5]
 [5 2 1]]
[0.67275604 0.94180287 0.24824571 0.94888115]
shuffled: [8 0 7 3 6 2 4 9 1 5]
choice: [4 2 7]


## Indexing and slicing arrays

In [7]:
A = np.arange(1, 13).reshape(3, 4)
print(A)

# Row/column slicing
print("rows 0..1, cols 1..3:", A[0:2, 1:3])

# Fancy indexing (with lists/arrays of indices)
rows = [0, 2]
cols = [1, 3]
print("A[rows, cols] ->", A[rows, cols])

# Boolean masking
mask = A % 2 == 0
print("even mask:", mask)
print("even elements:", A[mask])

# Assign via mask
B = A.copy()
B[B < 5] = 0
print("B with values < 5 zeroed:", B)


[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
rows 0..1, cols 1..3: [[2 3]
 [6 7]]
A[rows, cols] -> [ 2 12]
even mask: [[False  True False  True]
 [False  True False  True]
 [False  True False  True]]
even elements: [ 2  4  6  8 10 12]
B with values < 5 zeroed: [[ 0  0  0  0]
 [ 5  6  7  8]
 [ 9 10 11 12]]


## Array operations (elementwise & ufuncs)

In [8]:
x = np.array([1, 2, 3, 4], dtype=np.float64)
y = np.array([10, 20, 30, 40], dtype=np.float64)

print("x + y:", x + y)
print("x * y:", x * y)
print("x / 2:", x / 2)
print("x ** 2:", x ** 2)

# Universal functions (ufuncs)
print("np.sqrt(x):", np.sqrt(x))
print("np.log1p(x):", np.log1p(x))
print("np.maximum(x, 2):", np.maximum(x, 2))

# ufunc methods: reduce, accumulate
print("np.add.reduce(x):", np.add.reduce(x))
print("np.multiply.accumulate(x):", np.multiply.accumulate(x))


x + y: [11. 22. 33. 44.]
x * y: [ 10.  40.  90. 160.]
x / 2: [0.5 1.  1.5 2. ]
x ** 2: [ 1.  4.  9. 16.]
np.sqrt(x): [1.         1.41421356 1.73205081 2.        ]
np.log1p(x): [0.69314718 1.09861229 1.38629436 1.60943791]
np.maximum(x, 2): [2. 2. 3. 4.]
np.add.reduce(x): 10.0
np.multiply.accumulate(x): [ 1.  2.  6. 24.]


## Array operations II (type promotion & NaNs)

In [9]:
# Type promotion
i = np.array([1, 2, 3], dtype=np.int32)
f = np.array([0.5, 0.5, 0.5], dtype=np.float32)
print("result dtype:", (i + f).dtype)

# Handling NaNs
vals = np.array([1.0, np.nan, 3.0, np.nan])
print("np.isnan:", np.isnan(vals))
print("np.nan_to_num:", np.nan_to_num(vals, nan=-1.0))
print("np.nanmean:", np.nanmean(vals))


result dtype: float64
np.isnan: [False  True False  True]
np.nan_to_num: [ 1. -1.  3. -1.]
np.nanmean: 2.0


## Filtering arrays and modifying array values

In [10]:
data = rng.normal(loc=50, scale=10, size=20)
print("data:", np.round(data, 2))

# Filter by condition
high = data[data > 60]
print("> 60:", np.round(high, 2))

# Modify in place using mask
mask = data < 40
data[mask] = 40
print("clipped low to 40:", np.round(data, 2))


data: [47.   59.03 33.78 48.42 54.49 36.56 49.18 67.25 76.18 57.77 58.29 40.41
 37.91 35.88 55.42 57.52 43.41 37.71 52.58 53.13]
> 60: [67.25 76.18]
clipped low to 40: [47.   59.03 40.   48.42 54.49 40.   49.18 67.25 76.18 57.77 58.29 40.41
 40.   40.   55.42 57.52 43.41 40.   52.58 53.13]


## The `where()` function

In [11]:
temps = np.array([12, -5, 18, 0, 7, -2])
labels = np.where(temps < 0, 'freezing', 'non-freezing')
print(labels)

# Conditional replacement
cost = np.array([100, 250, 80, 120])
discounted = np.where(cost > 100, cost * 0.9, cost)
print(discounted)

# Combine conditions
arr = np.array([-3, -1, 0, 1, 3])
out = np.where(arr < 0, -1, 
               np.where(arr > 0, 1, 0))
print(out)


['non-freezing' 'freezing' 'non-freezing' 'non-freezing' 'non-freezing'
 'freezing']
[100. 225.  80. 108.]
[-1 -1  0  1  1]


## Array aggregation

In [12]:
A = rng.integers(0, 100, size=(4,5))
print(A)

print("sum (all):", A.sum())
print("sum axis=0 (column sums):", A.sum(axis=0))
print("mean axis=1 (row means):", A.mean(axis=1))
print("std (population):", A.std(ddof=0))

# Keep dimensions
print("row max keepdims:", A.max(axis=1, keepdims=True))


[[56 66 89 33 68]
 [90 45 25 29 33]
 [62 25 83 35  6]
 [ 0  2 62 34 28]]
sum (all): 871
sum axis=0 (column sums): [208 138 259 131 135]
mean axis=1 (row means): [62.4 44.4 42.2 25.2]
std (population): 26.72166723840412
row max keepdims: [[89]
 [90]
 [83]
 [62]]


## Array functions: arg*, unique, clip, percentiles

In [13]:
x = rng.integers(0, 10, size=10)
print("x:", x)
print("argmax:", x.argmax(), "argmin:", x.argmin())
print("unique values:", np.unique(x))
print("clip to [2,7]:", np.clip(x, 2, 7))
print("percentiles (25, 50, 75):", np.percentile(x, [25, 50, 75]))


x: [6 0 5 6 0 1 3 3 9 4]
argmax: 8 argmin: 1
unique values: [0 1 3 4 5 6 9]
clip to [2,7]: [6 2 5 6 2 2 3 3 7 4]
percentiles (25, 50, 75): [1.5  3.5  5.75]


## Sorting arrays

In [17]:
x = rng.integers(0, 100, size=10)
print("x:", x)
print("np.sort(x):", np.sort(x))    # returns a sorted copy
x.sort()                             # in-place sort
print("in-place sorted x:", x)

# Sort along axis
A = rng.integers(0, 100, size=(3,4))
print("A:", A)
print("sorted rows:", np.sort(A, axis=1))
print("sorted cols:", np.sort(A, axis=0))

# argsort to get ordered indices
idx = np.argsort(A[:, 0])
print("row indices sorted by first column:", idx)
print("A sorted by first column:", A[idx])


x: [71 24 60 83 46 18  7 86 50 17]
np.sort(x): [ 7 17 18 24 46 50 60 71 83 86]
in-place sorted x: [ 7 17 18 24 46 50 60 71 83 86]
A: [[29 75 37 61]
 [81 20 85 75]
 [33 24 34  8]]
sorted rows: [[29 37 61 75]
 [20 75 81 85]
 [ 8 24 33 34]]
sorted cols: [[29 20 34  8]
 [33 24 37 61]
 [81 75 85 75]]
row indices sorted by first column: [0 2 1]
A sorted by first column: [[29 75 37 61]
 [33 24 34  8]
 [81 20 85 75]]


## Aggregation and sorting (top-k & partition)

In [18]:
x = rng.integers(0, 100, size=15)
print("x:", x)

# Top-5 largest using argpartition (efficient)
k = 5
idx = np.argpartition(x, -k)[-k:]
print("indices of top-5:", idx)
print("top-5 values (unordered):", x[idx])

# Order those top-k
ordered_idx = idx[np.argsort(x[idx])][::-1]
print("top-5 ordered desc:", x[ordered_idx])


x: [42 61 77 53 10 63  9 17 96 24 57 68 82  8 84]
indices of top-5: [11  2 12 14  8]
top-5 values (unordered): [68 77 82 84 96]
top-5 ordered desc: [96 84 82 77 68]


## Vectorization

In [21]:
%%time
# Compute z = (x^2 + 2y) / (y + 1) for large arrays
N = 100_000  # 100k elements
x = rng.normal(size=N)
y = rng.normal(size=N)

# Vectorized
# %%timeit -n 3 -r 3
z = (x**2 + 2*y) / (y + 1)


CPU times: total: 0 ns
Wall time: 5 ms


In [22]:
%%timeit

# Non-vectorized (Python loop)
# %%timeit -n 1 -r 1
z_loop = np.empty_like(x)
for i in range(N):
    z_loop[i] = (x[i]**2 + 2*y[i]) / (y[i] + 1)


75.3 ms ± 2.69 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Broadcasting

In [20]:
# Example 1: Add a column vector to a row vector -> full matrix
col = np.array([[1], [2], [3]])   # shape (3,1)
row = np.array([10, 20, 30, 40])  # shape (4,)
M = col + row                     # result shape (3,4)
print(M)

# Example 2: Column-wise standardization
X = rng.normal(loc=0, scale=1, size=(5, 3))
mean = X.mean(axis=0, keepdims=True)
std = X.std(axis=0, ddof=0, keepdims=True)
X_std = (X - mean) / std
print("column means after standardization:", np.round(X_std.mean(axis=0), 6))

# Example 3: Pairwise distances using broadcasting
A = rng.normal(size=(4, 2))
B = rng.normal(size=(5, 2))
# Compute squared Euclidean distance: ||A_i - B_j||^2
# Shapes: A[:, None, :] -> (4,1,2); B[None, :, :] -> (1,5,2)
D2 = ((A[:, None, :] - B[None, :, :])**2).sum(axis=2)
print("distance matrix shape:", D2.shape)
print(np.round(D2, 3))


[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]
column means after standardization: [ 0.  0. -0.]
distance matrix shape: (4, 5)
[[10.28   1.496  6.06   7.961  3.731]
 [ 3.239  3.127  1.634  0.913  4.562]
 [ 2.343  0.534  0.905  2.915  0.126]
 [ 5.074  0.164  2.288  3.886  1.259]]


## Wrap-up & quick exercises

**Exercises (try before peeking at answers):**
1. Using `rng.integers`, create a `(6, 4)` array and replace values < 10 with 10 using masking.
2. Given a 1D array `a`, compute a z-score standardized version `(a - a.mean()) / a.std()`.
3. For a matrix `M (5x5)`, set its diagonal to 1 using `np.fill_diagonal`.
4. Use `np.where` to replace negative numbers with 0, numbers in `[0, 10)` with 1, and the rest with 2.
5. Generate two arrays `x, y` of length 200 and compute `(x * y) + np.sin(x)` vectorized.

*Stretch:* Implement top-3 values per row using `argpartition` along axis 1.
